## 1. Starter Pack: links, libraries, and configuration

Useful links:

- [Gemini Embeddings documentation](https://ai.google.dev/gemini-api/docs/embeddings)
- [Gemini document processing](https://ai.google.dev/gemini-api/docs/document-processing)
- [ChromaDB documentation](https://docs.trychroma.com/)
- [OpenAI Python SDK](https://github.com/openai/openai-python)
- [Google Colab secrets / userdata](https://colab.research.google.com/)

What this notebook uses:

- **Gemini Embedding 2** for embeddings
- **ChromaDB** for vector storage and search
- **PyMuPDF** to extract text from PDFs
- **Gemini Flash** to caption images
- **OpenAI** to generate the final RAG answer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/GenAI/Unstructured and Multimodal Data/Multimodal with Gemini

/content/drive/MyDrive/Freelance/DataHeroes/Course/GenAI/GenAI/Unstructured and Multimodal Data/Multimodal with Gemini


In [ ]:
!pip install -q google-genai chromadb pymupdf Pillow requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [ ]:
import os
import pathlib
import mimetypes
import requests
import pymupdf  # previously known as 'fitz'
import chromadb

from google import genai
from google.genai import types
from google.colab import userdata
from chromadb.config import Settings
from openai import OpenAI

In [ ]:
# -----------------------------
# API keys
# -----------------------------
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# -----------------------------
# Models
# -----------------------------
EMBEDDING_MODEL = "gemini-embedding-2"
GEMINI_TEXT_MODEL = "gemini-3.5-flash-lite"
OPENAI_MODEL = "gpt-5.6-luna"

# 768 is smaller and faster than the default 3072 dimensions.
EMBEDDING_DIM = 768

# -----------------------------
# ChromaDB
# -----------------------------
COLLECTION_NAME = "multimodal_rag"

# Helper Function

In [ ]:
IMAGE_MIME_MAP = {
    ".jpg":  "image/jpeg",
    ".jpeg": "image/jpeg",
    ".png":  "image/png",
    ".webp": "image/webp",
}

def load_bytes(path: str) -> bytes:
    """Load bytes from a URL or a local file."""
    if path.startswith("http"):
        response = requests.get(path, timeout=60)
        response.raise_for_status()
        return response.content
    return pathlib.Path(path).read_bytes()


## Renaming function
def suffix_of(path: str) -> str:
    """Extension of a path, ignoring any URL query string."""
    return pathlib.Path(path.split("?")[0]).suffix.lower()

### Embedding function





In [ ]:
## ----- Text Embedding -----

def embed_text(text: str) -> list[float]:
    """
    Embed text with Gemini Embedding 2.
    """

    # Create the text embedding
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(output_dimensionality=EMBEDDING_DIM),
    )

    return result.embeddings[0].values


## ----- Image Embedding -----

def embed_image(path: str) -> list[float]:
    """
    Create an embedding directly from an image with Gemini Embedding 2.
    """

    # Load the image and determine its file type
    image_bytes = load_bytes(path)
    mime = IMAGE_MIME_MAP.get(suffix_of(path), "image/jpeg")

    # Create the image embedding
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=[
            types.Part.from_bytes(
                data=image_bytes,
                mime_type=mime
            )
        ],
        config=types.EmbedContentConfig(output_dimensionality=EMBEDDING_DIM),
    )

    return result.embeddings[0].values


## ----- Video Embedding -----

def embed_video(url):
    """
    Create an embedding directly from a video with Gemini Embedding 2.
    """

    # Create the video embedding from its URL
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=[
            types.Part.from_uri(
                file_uri=url,
                mime_type="video/mp4"
            )
        ],
        config=types.EmbedContentConfig(output_dimensionality=EMBEDDING_DIM),
    )

    return response.embeddings[0].values

# Create ChromaDB

In [ ]:
# Create a persistent ChromaDB client.
# The database will be saved in the ./chroma_db folder.
chroma_client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=Settings(anonymized_telemetry=False),
)

# Create the collection if it doesn't already exist.
# Cosine similarity is used to compare embeddings.
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

# Confirm that the collection is ready
print("Collection ready:", COLLECTION_NAME)

Collection ready: multimodal_rag


In [ ]:
# to delete chroma db
# chroma_client.delete_collection(name=COLLECTION_NAME)

# PDF
PDFs already contain readable text, so we just split it into overlapping chunks and embed each one.

In [ ]:
# Urls to be ingested
META_REPORT_Q4 = "https://raw.githubusercontent.com/gladysmawarni/storage/main/Meta-Reports-Fourth-Quarter-and-Full-Year-2025-Results-2026.pdf"
META_REPORT_Q1 = "https://raw.githubusercontent.com/gladysmawarni/storage/main/Meta-03-31-2026-report.pdf"
NINTENDO_REPORT_Q3 = "https://raw.githubusercontent.com/gladysmawarni/storage/main/Nintendo-financial-result-03-2026.pdf"

In [ ]:
# Extract pdf content, for example purpose we limit it to just a few numbers of chunks instead of the whole documents
def extract_pdf_chunks(path: str, max_chunks: int = 10, chunk_size: int = 1200, overlap: int = 200) -> list[str]:
    """Extract text from a PDF and split it into overlapping chunks."""
    # Load the PDF as bytes
    pdf_bytes = load_bytes(path)

    # Open the PDF
    doc = pymupdf.open(stream=pdf_bytes, filetype="pdf")

    # Extract text from each page
    pages = []
    for page_number, page in enumerate(doc, start=1):
        text = page.get_text("text").strip()

        if text:
            pages.append(f"Page {page_number}\n{text}")

    # Combine all pages into one text string
    full_text = "\n\n".join(pages)

    # Split the text into smaller overlapping chunks
    chunks = []
    start = 0

    while start < len(full_text):
        end = start + chunk_size
        chunk = full_text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        # Move forward while keeping some text from the previous chunk
        start = end - overlap

    # Return only the requested number of chunks
    return chunks[:max_chunks]

In [ ]:
# Embed the chunks and store it in ChromaDB
def ingest_text(path, doc_id, metadata=None, max_chunks=10):
    metadata = metadata or {}
    title = metadata.get("title", doc_id)

    # Extract and split the PDF into smaller chunks
    chunks = extract_pdf_chunks(path, max_chunks)

    for i, chunk in enumerate(chunks):
        # Create an embedding for each text chunk
        formatted = f"title: {title} | text: {chunk}"
        embedding = embed_text(formatted)

        # Store the chunk, embedding, and metadata in ChromaDB
        collection.upsert(
            ids=[f"{doc_id}_{i}"],
            embeddings=[embedding],
            documents=[chunk],
            metadatas=[{
                "parent_id": doc_id,
                "chunk": i,
                "modality": "document",
                "source": path,
                **metadata
            }]
        )

    print(f"✅ {doc_id}: {len(chunks)} chunks ingested")

In [ ]:
# Ingest all three documents
ingest_text(
    path=META_REPORT_Q4,
    doc_id="pdf_meta_q4",
    metadata={"title": "Meta Q4 and Full Year 2025 Results", "topic": "finance"},
)
ingest_text(
    path=META_REPORT_Q1,
    doc_id="pdf_meta_q1",
    metadata={"title": "Meta Q1 2026 Results", "topic": "finance"},
)
ingest_text(
    path=NINTENDO_REPORT_Q3,
    doc_id="pdf_nintendo_q3",
    metadata={"title": "Nintendo Q3 FY2026 Results", "topic": "finance"},
)

✅ pdf_meta_q4: 5 chunks ingested
✅ pdf_meta_q1: 5 chunks ingested
✅ pdf_nintendo_q3: 5 chunks ingested


# Images

Images don't contain text, so we need two things:

- A **caption** from Gemini — the readable evidence the final LLM will read.
- An **embedding** — Gemini Embedding 2 can embed the raw image bytes directly, so the
  search vector represents the actual picture, not just the caption text.

In [ ]:
# url to be ingested
IMAGE_URL = "https://raw.githubusercontent.com/gladysmawarni/storage/main/phytoplankton-infographic.jpg"

In [ ]:
# Create a short searchable caption for an image

def caption_image(path: str) -> str:
    # Load the image and determine its file type
    image_bytes = load_bytes(path)
    mime = IMAGE_MIME_MAP.get(suffix_of(path), "image/jpeg")

    response = client.models.generate_content(
        model=GEMINI_TEXT_MODEL,
        contents=[
            "Describe this image for a RAG system. Be factual and concise.",
            types.Part.from_bytes(data=image_bytes, mime_type=mime),
        ],
    )
    return response.text.strip()

In [ ]:
# Embed the image and store it in ChromaDB together with its caption
def ingest_image(path: str, doc_id: str, metadata: dict | None = None) -> list[str]:
    """Add an image to ChromaDB with a Gemini-generated caption as evidence."""
    metadata = metadata or {}

    # create image caption and embed the image
    caption = caption_image(path)
    vec = embed_image(path)

    # Store the chunk, embedding, and metadata in ChromaDB
    collection.upsert(
        ids=[doc_id],
        embeddings=[vec],
        documents=[caption],
        metadatas=[{
            "parent_id": doc_id,
            "chunk": 0,
            "modality": "image",
            "source": path,
            **metadata,
        }],
    )
    print(f"✅ {doc_id}: image ingested")
    print("Caption:", caption[:250], "...")
    return [doc_id]

In [ ]:
# ingest an image of an infographic
ingest_image(
    path=IMAGE_URL,
    doc_id="img_001",
    metadata={"title": "Phytoplankton infographic", "topic": "phytoplankton"},
)

✅ img_001: image ingested
Caption: An infographic titled "PHYTOPLANKTON THE CORNERSTONE OF MARINE FOOD WEBS" explains the ecological importance of phytoplankton, created by Emily Heagney for the University of Guelph.

Key points include:
- **Definition:** Phytoplankton are microscopic ...


['img_001']

# Videos

In [ ]:
# Download a video from a public github
VIDEO_URL = "https://raw.githubusercontent.com/gladysmawarni/storage/main/videoplayback.mp4"
!wget -O datascience.mp4 {VIDEO_URL}

--2026-09-02 15:53:08--  https://raw.githubusercontent.com/gladysmawarni/storage/main/videoplayback.mp4
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4389599 (4.2M) [application/octet-stream]
Saving to: ‘datascience.mp4’

datascience.mp4     100%[===================>]   4.19M  25.7MB/s    in 0.2s    

2026-09-02 15:53:08 (25.7 MB/s) - ‘datascience.mp4’ saved [4389599/4389599]



In [ ]:
# Upload the video to Gemini
video = client.files.upload(file="datascience.mp4")

# Get the Gemini file URI
print(video.uri)

https://generativelanguage.googleapis.com/v1beta/files/sidkrepmc2aj


In [ ]:
# Create an embedding for the video
video_embedding = embed_video(video.uri)

# Store the video embedding and metadata in ChromaDB
collection.add(
    ids=["DS"],
    embeddings=[video_embedding],
    documents=["Video"],
    metadatas=[{
        "title": "Biggest Lie in Data Science Video",
        "source": video.uri,
        "modality": "video"
    }]
)

# Ask questions with RAG

```text
Question -> embed -> ChromaDB search -> retrieved evidence -> LLM answer
```

It prints the evidence it found (so you can see which modality it came from), then asks the LLM to answer using only that evidence.

In [ ]:
def ask(question, n_results=1):

    # Search ChromaDB
    formatted = f"task: question answering | query: {question}"
    query_embedding = embed_text(formatted)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

    metadata = results["metadatas"][0][0]
    document = results["documents"][0][0]

    print("Retrieved:", metadata)

    # Prepare the context
    if metadata["modality"] == "video":
        #  Add the video and question to the context
        context = [
            types.Part.from_uri(
                file_uri=metadata["source"],
                mime_type="video/mp4"
            ),
            question
        ]

    elif metadata["modality"] == "image":
        image = requests.get(metadata["source"])
        image.raise_for_status()
        # Add the image and question to the context
        context = [
            types.Part.from_bytes(
                data=image.content,
                mime_type="image/jpeg"
            ),
            question
        ]

    else:
        # Use the retrieved text as the context
        context = f"Context:\n{document}\n\nQuestion: {question}"

    # Ask Gemini
    response = client.models.generate_content(
        model=GEMINI_TEXT_MODEL,
        contents=context
    )

    print("Answer:", response.text)

    return response.text

In [ ]:
ask("What is phytoplankton?", n_results=1)

Retrieved: {'parent_id': 'img_001', 'modality': 'image', 'title': 'Phytoplankton infographic', 'chunk': 0, 'source': 'https://www.uoguelph.ca/research/innovation/sites/default/files/Infographics/Emily%20Heagney%20-%20Infographics%20in%20Practice.jpg', 'topic': 'phytoplankton'}
Answer: Based on the provided infographic, **phytoplankton are microscopic, plant-like aquatic organisms that perform photosynthesis**, just like plants on land. They serve as the cornerstone of marine food webs and are essential for nearly every ocean creature.


'Based on the provided infographic, **phytoplankton are microscopic, plant-like aquatic organisms that perform photosynthesis**, just like plants on land. They serve as the cornerstone of marine food webs and are essential for nearly every ocean creature.'

In [ ]:
ask("What is the biggest lie in data science?", n_results=1)

Retrieved: {'title': 'Biggest Lie in Data Science Video', 'source': 'https://generativelanguage.googleapis.com/v1beta/files/sidkrepmc2aj', 'modality': 'video'}
Answer: The biggest lie in data science according to this video is that deep learning models are always 'black boxes' that stakeholders can't trust.


"The biggest lie in data science according to this video is that deep learning models are always 'black boxes' that stakeholders can't trust."

In [ ]:
ask("How did Nintendo perform financially last year?", n_results=2)

Retrieved: {'topic': 'finance', 'source': 'https://www.nintendo.co.jp/ir/pdf/2026/260203_3e.pdf', 'parent_id': 'pdf_nintendo_q3', 'chunk': 0, 'title': 'Nintendo Q3 FY2026 Results', 'modality': 'document'}
Answer: Based on the provided document, Nintendo's financial performance for the first nine months of the previous fiscal year (**FY25/Q1-Q3**, covering the period up to December 31, 2024) was as follows:

* **Net sales:** 956.2 billion yen
* **Operating profit:** 247.5 billion yen
* **Operating profit ratio:** 25.9%
* **Ordinary profit:** 327.1 billion yen
* **Net profit (Profit attributable to owners of parent):** 237.1 billion yen
* **Net profit ratio:** 24.8%


"Based on the provided document, Nintendo's financial performance for the first nine months of the previous fiscal year (**FY25/Q1-Q3**, covering the period up to December 31, 2024) was as follows:\n\n* **Net sales:** 956.2 billion yen\n* **Operating profit:** 247.5 billion yen\n* **Operating profit ratio:** 25.9%\n* **Ordinary profit:** 327.1 billion yen\n* **Net profit (Profit attributable to owners of parent):** 237.1 billion yen\n* **Net profit ratio:** 24.8%"